# Backtest harness

`backtest()` in `src/backtest.py` runs any model function through a rolling-origin backtest, and `score()` turns the results into MAPE and MASE. This notebook checks the harness on a toy series and then runs the naive model through it.

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
from src.data import model_series
from src.backtest import backtest, score

In [2]:
def naive(history, horizon):
    return np.repeat(history.iloc[-1], horizon)

## Check on a toy series

In [3]:
toy = pd.Series(np.arange(60, dtype=float), index=pd.date_range("2024-01-01", periods=60))
toy_results = backtest(naive, toy, pd.Series(True, index=toy.index), first_origin="2024-02-01", step=7)
toy_results.head(3)

,origin,date,step,actual,forecast,scale,usable
0,2024-02-01,2024-02-01,1,31.0,30.0,7.0,True
1,2024-02-01,2024-02-02,2,32.0,30.0,7.0,True
2,2024-02-01,2024-02-03,3,33.0,30.0,7.0,True


In [4]:
score(toy_results)

MAPE       16.417088
MASE        1.071429
MAE         7.500000
origins     3.000000
dtype: float64

On a line going up by 1 a day, the naive forecast is the value from the day before the origin, so the errors over 14 days are 1 to 14 and the MAE is 7.5. The seasonal naive error on the history is 7 every day, so MASE should be 7.5 / 7 = 1.07, which is what the harness gives.

## Model series

In [5]:
daily = pd.read_parquet("../data/daily.parquet")
y, usable = model_series(daily)
len(y), (~usable).sum(), y.isna().sum()

(1095, np.int64(36), np.int64(0))

In [6]:
y[~usable].head(10)

date
2023-12-14    3440.5
2024-01-17    3262.5
2024-01-18    2733.0
2024-03-16    2127.0
2024-03-17    1721.5
2024-03-19    2436.5
2024-04-11    2454.5
2024-04-12    1908.0
2024-04-13    1077.0
2024-04-14    1189.5
Name: total, dtype: float64

## Naive model

In [7]:
results = backtest(naive, y, usable)
results["origin"].nunique(), results["origin"].min(), results["origin"].max(), results["usable"].sum()

(51,
 Timestamp('2024-11-04 00:00:00'),
 Timestamp('2026-10-05 00:00:00'),
 np.int64(700))

In [8]:
table = pd.DataFrame({"naive": score(results)}).T
table.round(3)

,MAPE,MASE,MAE,origins
naive,39.833,1.768,961.471,51.0


In [9]:
ok = results[results["usable"]]
mape_by_step = ((ok["actual"] - ok["forecast"]).abs() / ok["actual"]).groupby(ok["step"]).mean() * 100
mape_by_step.round(1)

step
1     44.2
2     43.2
3     42.1
4     42.3
5     38.2
6     27.8
7     27.4
8     49.1
9     48.5
10    49.3
11    45.7
12    40.2
13    30.0
14    28.4
dtype: float64

The naive model is off by 39.8% on average, with a MASE of 1.77. Every origin is a Monday, so it forecasts the next two weeks with the Sunday before, which is too low for weekdays (38 to 49% error) and closer on weekends (27 to 30%). That's the bar the baselines and models have to beat.